# 🧠 Reasoning Pruning: Interactive Data Creation & Exploration Laboratory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avrymi-asraf/reasoning-pruning-agy/blob/master/notebooks/01_explore_pruning.ipynb)

This notebook provides an **end-to-end, live interactive laboratory** for validating the complete **Data Creation Pipeline** of Reasoning Pruning. We ingest authentic benchmark questions across multi-domain cognitive families, generate reasoning traces with Generator model $G$, audit traces live for skippable overthinking spans using Decision Auditor $D$, render visual step diffs, extract $(x \to y)$ transition pairs, execute multi-depth recursive rollouts, build a versioned Hugging Face Dataset, validate all schema constraints, and synchronize directly with the Hugging Face Hub.

### 🎯 The Core Pipeline Flow:
$$\text{Benchmark Question } q \xrightarrow[\text{generate\_trace}]{\text{Generator } G} \text{Trace } (s_1..s_n) \xrightarrow[\text{find\_first\_skip}]{\text{Decision } D} \text{Skip } s_k \xrightarrow[\text{extract\_transition}]{\text{Pair}} (x \to y) \xrightarrow[\text{rollout\_pruning}]{\text{Recursive Rollout}} \text{PT Dataset}$$

| Pipeline Stage | Live Library Tool | Purpose | Output Contract |
|---|---|---|---|
| **1. Data Ingestion** | `rp.load_spectrum_benchmarks` | Streams real questions across 6 cognitive reasoning families | `List[Dict[str, Any]]` |
| **2. Trajectory Generation** | `rp.generate_trace` | Prompts generator $G$ and segments reasoning steps | `ReasoningTrace` |
| **3. Decision Auditing** | `rp.find_first_skip` | Audits trace live with decision model $D$ to find first safe skip | `PruneDecision` |
| **4. Visual Observability** | `rp.render_trace_diff` | Renders color-coded HTML diff of kept/pruned steps | `HTML` / `RichPanel` |
| **5. Transition Extraction** | `rp.extract_transition` | Isolates $(x \to y)$ local jump training pair | `TransitionExample` |
| **6. Recursive Rollout** | `rp.rollout_pruning` | Multi-depth iterative pruning and continuation | `RolloutResult` |
| **7. Dataset Assembly** | `rp.build_pt_dataset` | Assembles parallel Hugging Face Dataset across questions | `datasets.Dataset` |
| **8. Hub Synchronization** | `rp.push_dataset_to_hf` | Uploads dataset with automated lineage documentation | Hugging Face Hub URL |

# 1. Environment bootstrap (Google Colab vs local workspace)
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    print("🚀 Running on Google Colab. Setting up environment and repository...")
    REPO_DIR = "/content/reasoning-pruning-agy"
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/avrymi-asraf/reasoning-pruning-agy.git {REPO_DIR}
    %cd {REPO_DIR}
    !pip install -q -e .
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    print("✅ reasoning-pruning repository cloned and installed!")
else:
    print("💻 Running in local workspace.")

import json
import re
import pandas as pd
from IPython.display import HTML, display

# Import core reasoning_pruning tools (automatically initializes Colab secrets & .env)
import reasoning_pruning as rp
from reasoning_pruning.types import (
    ReasoningTrace,
    PruneDecision,
    TransitionExample,
    RolloutResult,
)

# Determine available provider and select optimal default models
MODEL_G = rp.get_default_generator_model()
MODEL_D = rp.get_default_decision_model()

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
gemini_key = os.environ.get("GEMINI_API_KEY")
openai_key = os.environ.get("OPENAI_API_KEY")

print(f"✅ reasoning_pruning v{rp.__version__} loaded successfully.")
print(f"🔑 HF_TOKEN configured: {bool(hf_token)}")
print(f"🔑 GEMINI_API_KEY configured: {bool(gemini_key)}")
print(f"🔑 OPENAI_API_KEY configured: {bool(openai_key)}")
print(f"🤖 Generator Model (G): {MODEL_G}")
print(f"⚖️  Decision Auditor (D): {MODEL_D}")


In [ ]:
# 1. Environment bootstrap (Google Colab vs local workspace)
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    print("🚀 Running on Google Colab. Setting up environment and repository...")
    REPO_DIR = "/content/reasoning-pruning-agy"
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/avrymi-asraf/reasoning-pruning-agy.git {REPO_DIR}
    %cd {REPO_DIR}
    !pip install -q -e .
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    
    # Auto-load secrets from Google Colab userdata
    try:
        from google.colab import userdata
        for secret_key in ["HF_TOKEN", "HUGGINGFACE_TOKEN", "GEMINI_API_KEY", "GEMINI_TOKEN", "OPENAI_API_KEY", "ANTHROPIC_API_KEY", "DEEPSEEK_API_KEY", "WANDB_API_KEY"]:
            try:
                val = userdata.get(secret_key)
                if val and secret_key not in os.environ:
                    os.environ[secret_key] = val
            except Exception:
                pass
        print("🔑 Colab secrets loaded into os.environ.")
    except (ImportError, ModuleNotFoundError):
        pass
    print("✅ reasoning-pruning repository cloned and installed!")
else:
    print("💻 Running in local workspace.")

# Load .env if present
for env_path in ["/content/.env", "/content/reasoning-pruning-agy/.env", ".env"]:
    if os.path.exists(env_path):
        try:
            from dotenv import load_dotenv
            load_dotenv(env_path, override=False)
        except ImportError:
            with open(env_path) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        os.environ.setdefault(k.strip(), v.strip())

# Map GEMINI_TOKEN alias to GEMINI_API_KEY for LiteLLM
if "GEMINI_TOKEN" in os.environ and "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = os.environ["GEMINI_TOKEN"]

import json
import re
import pandas as pd
from IPython.display import HTML, display

# Import core reasoning_pruning tools
import reasoning_pruning as rp
from reasoning_pruning.types import (
    ReasoningTrace,
    PruneDecision,
    TransitionExample,
    RolloutResult,
)

# Determine available provider and select optimal default models
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
gemini_key = os.environ.get("GEMINI_API_KEY")
openai_key = os.environ.get("OPENAI_API_KEY")

if "RP_MODEL_G" in os.environ:
    MODEL_G = os.environ["RP_MODEL_G"]
elif gemini_key:
    MODEL_G = "gemini/gemini-2.5-flash"
elif openai_key:
    MODEL_G = "gpt-4o-mini"
elif hf_token:
    MODEL_G = "huggingface/Qwen/Qwen2.5-7B-Instruct"
else:
    MODEL_G = "gemini/gemini-2.5-flash"

MODEL_D = os.environ.get("RP_MODEL_D", MODEL_G)

print(f"✅ reasoning_pruning v{rp.__version__} loaded successfully.")
print(f"🔑 HF_TOKEN configured: {bool(hf_token)}")
print(f"🔑 GEMINI_API_KEY configured: {bool(gemini_key)}")
print(f"🤖 Generator Model (G): {MODEL_G}")
print(f"⚖️  Decision Auditor (D): {MODEL_D}")

## 2. Stage 1: Load Multi-Domain Reasoning Benchmark Spectrum

We stream authentic benchmark questions across **6 distinct cognitive reasoning families** using `rp.load_spectrum_benchmarks`:
1. **Arithmetic & Word Math**: GSM8K (`openai/gsm8k`) & SVAMP (`ChilleD/SVAMP`)
2. **Scientific Reasoning**: ARC-Challenge (`allenai/ai2_arc`)
3. **Commonsense & Physical Logic**: CommonsenseQA (`tau/commonsense_qa`)
4. **Multi-hop & Deductive Logic**: HotpotQA (`hotpotqa/hotpot_qa`)
5. **Extractive & Span QA**: SQuAD 2.0 (`rajpurkar/squad_v2`)

In [ ]:
print("📥 Streaming live questions across multi-domain reasoning benchmarks...")
REAL_SPECTRUM = rp.load_spectrum_benchmarks(
    benchmarks=["gsm8k", "arc_challenge", "commonsense_qa", "hotpot_qa", "svamp", "squad_v2"],
    samples_per_benchmark=2,
    streaming=True,
)

print(f"✅ Successfully loaded {len(REAL_SPECTRUM)} benchmark questions across {len(set(x['category'] for x in REAL_SPECTRUM))} categories.\n")

df_spectrum = pd.DataFrame([
    {
        "ID": item["id"],
        "Dataset": item["dataset"],
        "Category": item["category"],
        "Question Stem": item["question"][:85] + ("..." if len(item["question"]) > 85 else ""),
        "Ground Truth": item["ground_truth"][:35] + ("..." if len(item["ground_truth"]) > 35 else ""),
    }
    for item in REAL_SPECTRUM
])

display(df_spectrum)

## 3. Stage 2: Live Reasoning Trace Generation (`rp.generate_trace`)

Prompts Generator model $G$ live via `rp.generate_trace` on an authentic benchmark problem. The output reasoning trajectory is automatically segmented into discrete, indexed deduction steps ($s_1, s_2, \dots, s_n$).

In [ ]:
# Select a sample question from the spectrum (GSM8K arithmetic problem)
sample_problem = REAL_SPECTRUM[0]
question_text = sample_problem["question"]
category_name = sample_problem["category"]
dataset_name = sample_problem["dataset"]

print(f"📌 Selected Benchmark: {dataset_name} ({category_name})")
print(f"🎯 Question:\n{question_text}\n")
print(f"🔑 Ground Truth: {sample_problem['ground_truth']}\n")
print(f"🚀 Generating live reasoning trajectory with Generator G ({MODEL_G})...")

# Live generation call
live_trace = rp.generate_trace(
    question=question_text,
    model=MODEL_G,
    temperature=0.7,
    max_tokens=1024,
)

print(f"\n✅ Generated {len(live_trace.steps)} reasoning steps ({live_trace.token_count} tokens):")
for i, step in enumerate(live_trace.steps):
    print(f"  [{i}] {step}")

## 4. Stage 3: Live Overthinking Audit (`rp.find_first_skip`)

Passes the generated `ReasoningTrace` to Decision Auditor $D$. The auditor determines whether intermediate steps contain conversational preambles, question restatements, redundant verification loops, or irrelevant detours that can be safely skipped without damaging logical correctness.

In [ ]:
print(f"⚖️ Auditing trace live with Decision Auditor ({MODEL_D})...")

# Live decision call
live_decision = rp.find_first_skip(
    trace=live_trace,
    decision_model=MODEL_D,
    temperature=0.0,
)

print("\n📋 Audit Decision Result:")
print(f"  Can Skip:        {live_decision.can_skip}")
if live_decision.can_skip:
    print(f"  Skip Span:       Steps [{live_decision.skip_start_idx} .. {live_decision.skip_end_idx}]")
    print(f"  Skipped Steps:   {live_decision.skipped_steps}")
    print(f"  Next Kept Step:  {live_decision.next_step}")
    print(f"  Auditor Reason:  {live_decision.reason}")
else:
    print(f"  Reason:          {live_decision.reason}")

## 5. Stage 4: Interactive Visual Trace Diff Rendering (`rp.render_trace_diff`)

Renders a transparent, color-coded HTML diff:
- 🟩 **Green text**: Kept context prefix ($x$)
- 🟥 **Red strikethrough**: Removable/redundant thoughts ($s_k$)
- 🟦 **Cyan text**: Next useful deduction target ($y$)

In [ ]:
# Render interactive HTML diff from live objects
html_diff = rp.render_trace_diff(live_trace, live_decision, as_html=True)
display(HTML(html_diff))

## 6. Stage 5: Local Transition Extraction (`rp.extract_transition`)

Transforms the audited trace into a concrete training transition example $(x \to y)$ where the model learns during generation to bypass the redundant span directly.

In [ ]:
if live_decision.can_skip:
    live_transition = rp.extract_transition(
        trace=live_trace,
        decision=live_decision,
        depth=1,
        example_id=f"{sample_problem['id']}_d1",
    )
    
    print("=" * 70)
    print(f"🎯 EXTRACTED PRUNING-TRANSITION TRAINING PAIR: {live_transition.id}")
    print("=" * 70)
    print(f"📌 Input Context (x):\n{live_transition.input_x}\n")
    print(f"🚀 Target Continuation (y):\n{live_transition.target_y}\n")
    print(f"✂️ Skipped Thoughts:\n{live_transition.skipped_steps}\n")
    print(f"💡 Audit Justification:\n{live_transition.skip_reason}")
else:
    print("No skippable steps found for this trace; transition extraction not applicable.")

## 7. Stage 6: Multi-Depth Recursive Rollouts on Complex Tasks (`rp.rollout_pruning`)

Executes recursive multi-depth pruning across iterations on a multi-step arithmetic problem. At each depth $d$, the pruned prefix is fed back into $G$, auditing the continuation for secondary redundancies until the final answer is reached.

In [ ]:
rollout_q = sample_problem["question"]

print(f"🔬 Running Live Multi-Depth Rollout on Problem:")
print(f"   {rollout_q}\n")

rollout_res = rp.rollout_pruning(
    question=rollout_q,
    generator_model=MODEL_G,
    decision_model=MODEL_D,
    max_depth=3,
)

print(f"\n✅ Rollout complete! Total Depths: {len(rollout_res.traces)}, Transitions Extracted: {len(rollout_res.transitions)}")
print(f"Original Steps: {rollout_res.original_step_count} ➔ Final Steps: {rollout_res.final_step_count} ({rollout_res.compression_ratio*100:.1f}% reduction)")

for i, tr in enumerate(rollout_res.transitions, 1):
    print(f"\n--- [Transition {i} | Depth {tr.depth}] ---")
    print(f"Input (x):  {tr.input_x[:80]}...")
    print(f"Target (y): {tr.target_y}")
    print(f"Skipped:    {tr.skipped_steps}")
    print(f"Reason:     {tr.skip_reason}")

## 8. Stage 7: Full Dataset Construction (`rp.build_pt_dataset`)

Converts a collection of authentic benchmark questions across reasoning families into a complete Hugging Face `Dataset` ready for 4-bit QLoRA SFT training.

In [ ]:
# Select a representative batch of benchmark questions across categories
benchmark_questions = REAL_SPECTRUM[:4]

print(f"🔨 Building Hugging Face PT Dataset live across {len(benchmark_questions)} real benchmark questions...")
hf_dataset = rp.build_pt_dataset(
    questions=benchmark_questions,
    generator_model=MODEL_G,
    decision_model=MODEL_D,
    max_depth=2,
    max_workers=2,
)

print(f"\n✅ Built Live Dataset with {len(hf_dataset)} transition examples!")
if len(hf_dataset) > 0:
    df_ds = hf_dataset.to_pandas()
    display(df_ds[["id", "depth", "input_x", "target_y", "skip_reason"]].head(6))

## 9. Stage 8: Dataset Validation, Schema & Integrity Constraints

Validates the generated dataset against strict quality criteria:
1. **Schema Integrity**: All required columns present (`id`, `question`, `input_x`, `target_y`, `depth`, `skipped_steps`, `skip_reason`, `metadata`).
2. **Content Validity**: Zero empty inputs (`input_x`) or targets (`target_y`).
3. **Category & Depth Distribution**: Inspects transitions by reasoning family and pruning depth.

In [ ]:
print("🔍 Validating Dataset Integrity and Quality Constraints...")

# 1. Schema check
expected_cols = ["id", "question", "input_x", "target_y", "depth", "skipped_steps", "skip_reason", "metadata"]
for col in expected_cols:
    assert col in hf_dataset.column_names, f"Missing required column: {col}"
print("  ✅ All required schema columns present.")

# 2. Content validity
df_val = hf_dataset.to_pandas()
assert (df_val["input_x"].str.len() > 0).all(), "Found empty input_x entries!"
assert (df_val["target_y"].str.len() > 0).all(), "Found empty target_y entries!"
print(f"  ✅ All {len(df_val)} rows have valid non-empty input contexts and target continuations.")

# 3. Distribution metrics
category_counts = df_val["metadata"].apply(lambda m: m.get("category", "Unknown") if isinstance(m, dict) else "Unknown").value_counts()
depth_counts = df_val["depth"].value_counts()

print("\n📊 Transition Distribution by Category:")
for cat, count in category_counts.items():
    print(f"  - {cat}: {count} transitions")

print("\n📈 Transition Distribution by Depth:")
for d, count in depth_counts.items():
    print(f"  - Depth {d}: {count} transitions")

## 10. Stage 9: Hugging Face Hub Synchronization & Lineage Documentation

Pushes the validated dataset to the Hugging Face Hub using `rp.push_dataset_to_hf` with an automated Dataset Card documenting model provenance and depth breakdown. Then verifies round-trip download via `rp.load_pt_dataset`.

In [ ]:
if hf_token and len(hf_dataset) > 0:
    # Configure repository ID (e.g. username/rp-dataset-validation)
    repo_id = os.environ.get("RP_HF_REPO", "avreymi/rp-test-validation-v1")
    
    print(f"🚀 Pushing validated dataset ({len(hf_dataset)} rows) to Hugging Face Hub: {repo_id}...")
    hub_url = rp.push_dataset_to_hf(
        dataset=hf_dataset,
        repo_id=repo_id,
        private=True,
        token=hf_token,
        generator_model=MODEL_G,
        decision_model=MODEL_D,
        description="Validated multi-domain reasoning pruning dataset.",
    )
    print(f"\n✅ Successfully published to Hugging Face: {hub_url}")
    
    print("\n📥 Verifying roundtrip reload from Hugging Face Hub...")
    reloaded_ds = rp.load_pt_dataset(repo_id, split="train", token=hf_token)
    print(f"✅ Roundtrip verified! Reloaded {len(reloaded_ds)} transition examples from {repo_id}.")
else:
    print("ℹ️ HF_TOKEN not configured or empty dataset; skipping live Hub upload.")

## 11. Cross-Domain Overthinking Audit Comparison

Compare how overthinking patterns manifest across different cognitive reasoning families (e.g., Conversational Preamble in word math vs. Question Restatement or Redundant Fact Enumeration in science QA).

In [ ]:
print("🔬 Probing overthinking across distinct cognitive families:\n")
audit_summary = []

for item in REAL_SPECTRUM[:4]:
    trace = rp.generate_trace(question=item["question"], model=MODEL_G, temperature=0.7, max_tokens=256)
    decision = rp.find_first_skip(trace=trace, decision_model=MODEL_D, temperature=0.0)
    
    audit_summary.append({
        "Dataset": item["dataset"],
        "Category": item["category"],
        "Steps": len(trace.steps),
        "Can Skip": decision.can_skip,
        "Skipped Span": f"[{decision.skip_start_idx}..{decision.skip_end_idx}]" if decision.can_skip else "None",
        "Reason": decision.reason[:85] + "..." if len(decision.reason) > 85 else decision.reason,
    })

display(pd.DataFrame(audit_summary))